## Methodology

### Prediction target
Predicting P(YES) for each market, then comparing the model's prediction
against `prev_day_price` (the market's own 1-day-before-resolution price).
Large disagreements between model and market = mispricing candidates.

### Features
- `prev_day_price`: observed at prediction time. Doesn't leak. This is the
  market's own forecast — the thing we're trying to improve on.
- `category` (one-hot encoded): known at market creation. Doesn't leak.
  Day 2 showed Sports and Crypto are the miscalibrated categories, so this
  feature is critical.
- `volume`: total market volume, known at prediction time. Doesn't leak.
  Including it because high-volume markets may be better-priced (more
  participants → better information aggregation), so the model can learn
  to weight the market's signal more when volume is high.
- `lifetime_days`: known at prediction time. Technically off by roughly a day
  from the full lifetime (we'd know "current lifetime" not final), but the
  difference is negligible.

### Cross-validation
- Sort markets chronologically by `closed_time`. First 80% are used for training,
  last 20% for testing. Simulates predicting future markets from past data.
- I will also compare category distribution in train
  vs. test. We will need to ensure that this distribution is pretty similar

### Models
- **Logistic regression** as baseline — interpretable, but can't capture
  interactions like (Sports × high price) without manual feature engineering.
- **Random forest** as the main model — captures category × price
  interactions automatically, which is the exact pattern I noticed
  as the main source of mispricing.
- Both models trained on the same train split, evaluated on the same test.

### Evaluation metrics
- **Aggregate Brier** on the holdout. Baseline to beat
  is the market's own Brier of 0.162. If the model's Brier isn't lower
  than that on the test set, we haven't found mispricing.
- **Log loss** on the holdout — secondary metric
- **Per-category Brier** — aggregate Brier might hide that the model is
  great at Politics but useless at Sports, where the actual mispricing
  lives.
- **Calibration plot** on the holdout, with model and market overlaid.
- **Direct head-to-head**: compute Brier and log loss for (a) the market's
  `prev_day_price`, (b) the model's prediction, on identical test rows.